# CDVAEのMEGNet Perov強化&大きい結晶（原子数>20）も含めたデータセット

In [7]:
import torch
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
import glob

%matplotlib inline

In [8]:
result_dir = '/home/fujii/cdvae_comparison/main_results/megnet_perov_huge/'
os.makedirs(result_dir, exist_ok=True)

## CDVAEモデルの学習結果
- lr=0.00100に決定

In [9]:
api = wandb.Api()

In [10]:
urls = [
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/lvt2n6p4', # 1e-5はロスが振動しているので採用不可
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/dlo2co6w',
    'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/03n2a4ds',
    #'akihiro-fujii-university-of-tokyo/crystal_generation_mit/runs/13w8iowr',

]
lrs = [
    1e-5,
    1e-4,
    1e-3,
    #1e-5,
]

In [11]:
all_results = []
for lr, url in zip(lrs, urls):
    run = api.run(url)
    # ログされた履歴を取得
    history = run.history()

    # val_lossの最小値とそのステップを取得
    min_step = history['val_loss'].idxmin()
    min_val_loss = history['val_loss'][min_step]
    val_natom_loss = history['val_natom_loss'][min_step]
    val_natom_accuracy = history['val_natom_accuracy'][min_step]
    val_lattice_loss = history['val_lattice_loss'][min_step]
    val_eform_mae = history['val_eform_mae'][min_step]
    val_gap_mae = history['val_gap_mae'][min_step]
    val_tolerance_acc = history['val_tolerance_acc'][min_step]
    val_100more_acc = history['val_100more_acc'][min_step]

    all_results.append({
        'lr': lr,
        'min_val_loss': min_val_loss,
        'val_natom_loss': val_natom_loss,
        'val_natom_accuracy': val_natom_accuracy,
        'val_lattice_loss': val_lattice_loss,
        'val_eform_mae': val_eform_mae,
        'val_gap_mae': val_gap_mae,
        'val_tolerance_acc': val_tolerance_acc,
        'val_100more_acc': val_100more_acc,
    })

In [12]:
df = pd.DataFrame(all_results)
df.to_csv(os.path.join(result_dir, 'forward_result.csv'), index=False)
display(df)

,lr,min_val_loss,val_natom_loss,val_natom_accuracy,val_lattice_loss,val_eform_mae,val_gap_mae,val_tolerance_acc,val_100more_acc
0,0.00001,11.048626,1.923536,0.711568,0.499279,0.118947,0.740008,0.997830,1.0
1,0.00010,8.413797,2.419589,0.782282,0.305680,0.070745,0.676675,0.996962,1.0
2,0.00100,8.219883,2.173669,0.803263,0.266845,0.067685,0.663876,0.998264,1.0


## 最適化推論の学習率調整
- 推論コマンドの例: 
    - ```poetry run python scripts/evaluate.py --tasks opt --label lr00001 --lr 0.0001 --num_starting_points 128 --model_path /home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/shin_megnet_lr5e-6/ --target_bg 2.5 --num_saved_crys 0 --megnet_loss_mode True --coef_e_form 0 ```

In [ ]:
result_dir = '/home/fujii/cdvae_comparison/hydra/singlerun/2025-04-15/shin_megnet_lr5e-6/'
#os.listdir(result_dir)

In [ ]:
bandgap_margin = 0.2
raw_result_dict_list = []
decoded_result_dict_list = []
result_dict_list = []
for pred_path in glob.glob(os.path.join(result_dir,'eval_opt_megnet__bg2.50__*.pt')):
    d = torch.load(pred_path)
    d.keys()
    # 最適化に使った学習率の情報を得る
    lr_string = pred_path.split("eval_opt_lr")[-1].split(".")[0]
    #lr = float("0."+lr_string[1:])

    # ファイル名から学習率などを取得
    tag_dict = {}
    tag_name_list = ['lr','bg','grad-steps']
    for tag_data in os.path.basename(pred_path).split("__"):
        for tag_name in tag_name_list:
            if tag_data.startswith(tag_name):
                if tag_name == 'grad-steps':
                    tag_dict[tag_name] = int(tag_data.replace(tag_name,"").replace('.pt',''))
                else:
                    tag_dict[tag_name] = float(tag_data.replace(tag_name,"").replace('.pt',''))

    # band gap の予測値を得る
    raw_z_bandgap_pred = d['prediction_matched'][:,0]
    decoded_z_bandgap_pred = d['prediction_decoded'][:,0]
    assert raw_z_bandgap_pred.shape == decoded_z_bandgap_pred.shape
    ## 128になるまで、torch.nanを追加する
    if raw_z_bandgap_pred.shape[0] < 128:
        raw_z_bandgap_pred = torch.cat([raw_z_bandgap_pred, torch.full((128-raw_z_bandgap_pred.shape[0],), float('nan'))])
    if decoded_z_bandgap_pred.shape[0] < 128:
        decoded_z_bandgap_pred = torch.cat([decoded_z_bandgap_pred, torch.full((128-decoded_z_bandgap_pred.shape[0],), float('nan'))])
    # 予測バンドギャップとターゲットの差分をとる
    raw_z_bg_satisfaction = torch.clip(torch.abs(raw_z_bandgap_pred - tag_dict['bg'])-bandgap_margin, min=0).numpy() == 0
    decoded_z_bg_satisfaction = torch.clip(torch.abs(decoded_z_bandgap_pred - tag_dict['bg'])-bandgap_margin, min=0).numpy() == 0
    result_dict_list.append({
        'lr': tag_dict['lr'],
        'bg': tag_dict['bg'],
        'bg-margin': bandgap_margin,
        'grad_steps': tag_dict['grad-steps'],
        'raw_z_bg_satisfaction_rate': raw_z_bg_satisfaction.sum()/raw_z_bg_satisfaction.shape[0],
        'decoded_z_bg_satisfaction_rate': decoded_z_bg_satisfaction.sum()/decoded_z_bg_satisfaction.shape[0],
    })
result_df = pd.DataFrame(result_dict_list).set_index(['bg','bg-margin','lr','grad_steps']).sort_index()
result_df

In [ ]:
# best result
display(result_df.sort_values('decoded_z_bg_satisfaction_rate',ascending=False).head(1))
bg, margin, best_lr, best_steps = result_df.sort_values('decoded_z_bg_satisfaction_rate',ascending=False).head(1).index.values[0]